#### **Exporting to TensorRT**

Right now, the pipeline runs the model file (`best.pt`) directly 
through PyTorch, the same way it would run on a laptop or cloud 
server. This works, but it's not optimized for this specific chip.

**TensorRT** is NVIDIA's tool for compiling a model specifically for 
the exact GPU it'll run on,  in this case, the Jetson Orin Nano's 
GPU. Instead of interpreting the model's instructions on the fly 
(what PyTorch does), TensorRT pre-compiles a version tailored to 
this hardware's exact capabilities, typically resulting in a large 
speed improvement with little to no accuracy loss.

This is a one-time compilation step, it takes a few minutes, but 
only needs to happen once per model per device.

In [28]:
from ultralytics import YOLO
import config

model = YOLO(config.MODEL_PATH)
model.export(format="engine", device=0)

Ultralytics 8.4.86 🚀 Python-3.12.3 torch-2.12.1+cu130 CUDA:0 (Orin, 7485MiB)


Model summary (fused): 73 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/home/emertxe/Desktop/Emertxe/Shreya/helmet-detection-violation/best.pt' with input shape (1, 3, 1280, 1280) BCHW and output shape(s) (1, 9, 33600) (64.1 MB)

ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnxslim 0.1.94...


2026-07-27 15:51:26.500703482 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card1": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card1/device/vendor"
2026-07-27 15:51:26.501414220 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


ONNX: export success ✅ 2.4s, saved as '/home/emertxe/Desktop/Emertxe/Shreya/helmet-detection-violation/best.onnx' (43.2 MB)

TensorRT: starting export with TensorRT 11.1.0.106...
[07/27/2026-15:51:27] [TRT] [I] [MemUsageChange] Init CUDA: CPU +0, GPU +0, now: CPU 1820, GPU 6404 (MiB)
[07/27/2026-15:51:28] [TRT] [I] ----------------------------------------------------------------
[07/27/2026-15:51:28] [TRT] [I] Input filename:   /home/emertxe/Desktop/Emertxe/Shreya/helmet-detection-violation/best.onnx
[07/27/2026-15:51:28] [TRT] [I] ONNX IR version:  0.0.8
[07/27/2026-15:51:28] [TRT] [I] Opset version:    18
[07/27/2026-15:51:28] [TRT] [I] Producer name:    pytorch
[07/27/2026-15:51:28] [TRT] [I] Producer version: 2.12.1
[07/27/2026-15:51:28] [TRT] [I] Domain:           
[07/27/2026-15:51:28] [TRT] [I] Model version:    0
[07/27/2026-15:51:28] [TRT] [I] Doc string:       
[07/27/2026-15:51:28] [TRT] [I] ----------------------------------------------------------------
TensorRT: input "im

PosixPath('/home/emertxe/Desktop/Emertxe/Shreya/helmet-detection-violation/best.engine')

#### **The TensorRT engine : exported**

Exporting took about 4.4 minutes, a one-time cost. This produced 
`best.engine`, a version of the model compiled specifically for this 
Jetson Orin Nano's GPU. Let's now run the exact same full-video 
pipeline against this engine file instead of the original `.pt` 
file, and compare.

In [29]:
from main import run_pipeline, save_outputs
import time

start = time.time()
manager_trt, observations_trt, plate_matches_trt, sample_frames_trt = run_pipeline(
    model_path="best.engine"
)
elapsed_trt = time.time() - start

print(f"\nTotal time (TensorRT): {elapsed_trt:.1f} seconds")
print(f"FPS (TensorRT): {1500 / elapsed_trt:.2f}")
print(f"\nFor comparison, PyTorch baseline: 175.8 seconds, {1500/175.8:.2f} FPS")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Loading best.engine for TensorRT inference...
[07/27/2026-15:56:34] [TRT] [I] Loaded engine size: 44 MiB
[07/27/2026-15:56:34] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +135, now: CPU 0, GPU 177 (MiB)
Processed 1500 frames from: /home/emertxe/Desktop/Emertxe/Shreya/helmet-detection-violation/sample_videos/20211125082353_0060.mp4
Model: best.engine
  total_instances: 110
  confident_helmet_status: 36
  confirmed_violations: 13

Total time (TensorRT): 173.7 seconds
FPS (TensorRT): 8.64

For comparison, PyTorch baseline: 175.8 seconds, 8.53 FPS


### **Trying FP16 instead of FP32**

The engine we exported used FP32 (full precision), the same 
precision PyTorch uses by default. TensorRT's typical speedup often 
comes specifically from switching to FP16 (half precision), which 
does less work per calculation at a small, usually negligible, 
accuracy cost. Let's export an FP16 version and see if that closes 
the gap.

In [30]:
from ultralytics import YOLO
import config

model = YOLO(config.MODEL_PATH)
model.export(format="engine", device=0, half=True)

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.86 🚀 Python-3.12.3 torch-2.12.1+cu130 CUDA:0 (Orin, 7485MiB)
Model summary (fused): 73 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/home/emertxe/Desktop/Emertxe/Shreya/helmet-detection-violation/best.pt' with input shape (1, 3, 1280, 1280) BCHW and output shape(s) (1, 9, 33600) (64.1 MB)

ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 2.3s, saved as '/home/emertxe/Desktop/Emertxe/Shreya/helmet-detection-violation/best.onnx' (43.2 MB)

TensorRT: starting export with TensorRT 11.1.0.106...
requirements: Ultralytics requirement ['nvidia-modelopt[onnx]>=0.44'] not found, attempting AutoUpdate...
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: fin

PosixPath('/home/emertxe/Desktop/Emertxe/Shreya/helmet-detection-violation/best.engine')

### **Re-exporting with FP16**

The first export used FP32, keeping the same precision as PyTorch,
which explains why it gave no speedup. This time we export with 
FP16 (half precision), which does less computation per operation.

The export process automatically converted 229 of 231 model layers 
to FP16, keeping 2 layers at higher precision where FP16 would have 
lost too much numeric range, a built-in safety check rather than 
converting everything blindly.

In [31]:
from main import run_pipeline, save_outputs
import time

start = time.time()
manager_fp16, observations_fp16, plate_matches_fp16, sample_frames_fp16 = run_pipeline(
    model_path="best.engine"
)
elapsed_fp16 = time.time() - start

print(f"\nTotal time (TensorRT FP16): {elapsed_fp16:.1f} seconds")
print(f"FPS (TensorRT FP16): {1500 / elapsed_fp16:.2f}")
print(f"\nComparison so far:")
print(f"  PyTorch (.pt):        175.8s, {1500/175.8:.2f} FPS")
print(f"  TensorRT FP32:        173.7s, {1500/173.7:.2f} FPS")
print(f"  TensorRT FP16:        {elapsed_fp16:.1f}s, {1500/elapsed_fp16:.2f} FPS")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Loading best.engine for TensorRT inference...
[07/27/2026-16:11:17] [TRT] [I] Loaded engine size: 23 MiB
[07/27/2026-16:11:17] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +67, now: CPU 0, GPU 88 (MiB)
Processed 1500 frames from: /home/emertxe/Desktop/Emertxe/Shreya/helmet-detection-violation/sample_videos/20211125082353_0060.mp4
Model: best.engine
  total_instances: 112
  confident_helmet_status: 39
  confirmed_violations: 14

Total time (TensorRT FP16): 118.2 seconds
FPS (TensorRT FP16): 12.69

Comparison so far:
  PyTorch (.pt):        175.8s, 8.53 FPS
  TensorRT FP32:        173.7s, 8.64 FPS
  TensorRT FP16:        118.2s, 12.69 FPS


In [32]:
matched_fp16, total_fp16 = check_detection_recall(
    config.VIDEO_PATH, gt_df, "best.engine", config.TRACKER, config.CLASS_NAMES
)

print("Detection recall comparison (PyTorch baseline vs TensorRT FP16):\n")
for label in matched_fp16:
    recall_pt = matched[label] / total[label]
    recall_fp16 = matched_fp16[label] / total_fp16[label]
    print(f"{label}:")
    print(f"  PyTorch:      {matched[label]}/{total[label]} = {recall_pt:.1%}")
    print(f"  TensorRT FP16: {matched_fp16[label]}/{total_fp16[label]} = {recall_fp16:.1%}")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Loading best.engine for TensorRT inference...
[07/27/2026-16:15:33] [TRT] [I] Loaded engine size: 23 MiB
[07/27/2026-16:15:34] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +67, now: CPU 0, GPU 88 (MiB)
Detection recall comparison (PyTorch baseline vs TensorRT FP16):

rider:
  PyTorch:      1048/1552 = 67.5%
  TensorRT FP16: 1049/1552 = 67.6%
motorcycle:
  PyTorch:      894/1749 = 51.1%
  TensorRT FP16: 902/1749 = 51.6%


| | PyTorch | TensorRT FP16 |
|---|---|---|
| Rider detection recall | 67.5% | 67.6% |
| Motorcycle detection recall | 51.1% | 51.6% |

Detection recall is essentially unchanged, the tiny differences 
(+1 rider, +8 motorcycles, both under 1%) are well within normal 
run-to-run noise, not a real accuracy loss. Combined with the 49% 
speed improvement, this means **FP16 TensorRT export gives a real 
speedup on this hardware with no meaningful accuracy cost** for this 
model.

This is the clearest evidence in this whole notebook for the value 
of edge-specific optimization: the same model, the same weights, 
the same detection quality just compiled for this exact chip
runs in roughly two-thirds the time.